#  Modelos alternativos
Este avance implica construir múltiples modelos (individuales, no ensambles) relevantes para resolver el problema y evaluar su desempeño. Diferentes algoritmos pueden comportarse de manera óptima en diferentes tipos de datos o tareas. La construcción de modelos alternativos permite explorar y evaluar cuál de ellos proporciona el mejor rendimiento para un problema particular.

Además, los modelos se pueden ajustar para determinar si se puede mejorar su rendimiento. Diferentes configuraciones de hiperparámetros pueden afectar significativamente el rendimiento de un modelo. Construir modelos alternativos implica explorar y ajustar estos hiperparámetros para encontrar la configuración óptima.

Las siguientes son acciones que deberás abordar en este avance:

* Construir al menos 6 modelos diferentes (individuales, no ensambles), utilizando algoritmos variados.
* Comparar el rendimiento de los modelos obtenidos.
* Seleccionar los dos modelos que proporcionen el mejor rendimiento.
* Ajustar los dos mejores modelos.
* Elegir el modelo individual final.

# Modelos
Ahcer preparamiento de datos de entrada para modelos

In [6]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, GroupKFold

RANDOM_STATE = 42
TEST_SIZE = 0.25

# Ingresa los archivos que usó la preparación de datos de tu libreta.
annotations_path=Path("../DataCapture/anotaciones_baseline.csv")
heads_path=Path("../DataCapture/mediciones_mascara_rostro.csv")

IMAGE_COL = "imagen"
X_COL, Y_COL, W_COL, H_COL, ImageWidth, ImageHeight = "x", "y", "w", "h", "image_width", "image_height"

annotations = pd.read_csv(annotations_path)
heads = pd.read_csv(heads_path)

# Verificar que existan las columnas esperadas.
required_annotations = {IMAGE_COL, X_COL, Y_COL, W_COL, H_COL}
required_heads = {IMAGE_COL, X_COL, Y_COL, W_COL, H_COL}

missing_annotations = required_annotations - set(annotations.columns)
missing_heads = required_heads - set(heads.columns)

if missing_annotations:
    raise ValueError(
        f"Faltan columnas en anotaciones: {sorted(missing_annotations)}"
    )
if missing_heads:
    raise ValueError(
        f"Faltan columnas en mediciones de cabeza: {sorted(missing_heads)}"
    )

# Normalizar el identificador de imagen para que coincida entre archivos.
annotations[IMAGE_COL] = annotations[IMAGE_COL].astype(str).map(
    lambda value: Path(value).name
)
heads[IMAGE_COL] = heads[IMAGE_COL].astype(str).map(
    lambda value: Path(value).name
)

# Convertir coordenadas a valores numéricos y descartar filas incompletas.
for col in [X_COL, Y_COL, W_COL, H_COL]:
    annotations[col] = pd.to_numeric(annotations[col], errors="coerce")
    heads[col] = pd.to_numeric(heads[col], errors="coerce")

annotations = annotations.dropna(
    subset=[IMAGE_COL, X_COL, Y_COL, W_COL, H_COL]
).copy()
heads = heads.dropna(
    subset=[IMAGE_COL, X_COL, Y_COL, W_COL, H_COL]
).copy()

# Ignorar cajas con ancho o alto no válidos.
annotations = annotations[
    (annotations[W_COL] > 0) & (annotations[H_COL] > 0)
].copy()
heads = heads[
    (heads[W_COL] > 0) & (heads[H_COL] > 0)
].copy()

# Usa las dimensiones de cada imagen si están en el CSV de anotaciones.
if {"image_width", "image_height"}.issubset(annotations.columns):
    annotations["img_w"] = pd.to_numeric(
        annotations["image_width"], errors="coerce"
    )
    annotations["img_h"] = pd.to_numeric(
        annotations["image_height"], errors="coerce"
    )
    annotations = annotations.dropna(subset=["img_w", "img_h"])
else:
    # La libreta recuperada asumía imágenes de 640 × 480.
    # Cambia estos valores si tus imágenes tienen otras dimensiones.
    IMG_W = int(input("Ancho de imagen en píxeles (por ejemplo, 640): "))
    IMG_H = int(input("Alto de imagen en píxeles (por ejemplo, 480): "))
    annotations["img_w"] = IMG_W
    annotations["img_h"] = IMG_H

if (annotations["img_w"] <= 0).any() or (annotations["img_h"] <= 0).any():
    raise ValueError("Se encontraron dimensiones de imagen inválidas.")

# Renombrar las columnas de la cabeza para no confundirlas con las del objeto.
heads = heads[[IMAGE_COL, X_COL, Y_COL, W_COL, H_COL]].rename(
    columns={
        X_COL: "head_x",
        Y_COL: "head_y",
        W_COL: "head_w",
        H_COL: "head_h",
    }
)

heads["head_cx"] = heads["head_x"] + heads["head_w"] / 2
heads["head_cy"] = heads["head_y"] + heads["head_h"] / 2

# Centro de cada objeto anotado.
annotations = annotations.reset_index(drop=True)
annotations["_object_id"] = np.arange(len(annotations))
annotations["obj_cx"] = annotations[X_COL] + annotations[W_COL] / 2
annotations["obj_cy"] = annotations[Y_COL] + annotations[H_COL] / 2

# Asociar cada objeto con la cabeza más cercana de la misma imagen.
pairs = annotations[
    ["_object_id", IMAGE_COL, "obj_cx", "obj_cy"]
].merge(heads, on=IMAGE_COL, how="inner")

if pairs.empty:
    raise ValueError(
        "No se encontraron imágenes coincidentes entre anotaciones y cabezas."
    )

pairs["distancia"] = np.hypot(
    pairs["obj_cx"] - pairs["head_cx"],
    pairs["obj_cy"] - pairs["head_cy"]
)

closest_rows = pairs.groupby("_object_id")["distancia"].idxmin()
nearest_heads = pairs.loc[
    closest_rows,
    ["_object_id", "head_x", "head_y", "head_w", "head_h", "head_cx", "head_cy"]
]

df = annotations.merge(nearest_heads, on="_object_id", how="inner")

# Característica de hora extraída del nombre de imagen, como en la libreta.
hour_text = df[IMAGE_COL].str.extract(r"_(\d{2})\d{4}_", expand=False)
if hour_text.isna().any():
    examples = df.loc[hour_text.isna(), IMAGE_COL].head().tolist()
    raise ValueError(
        "No pude extraer la hora de algunos nombres de imagen. "
        f"Revisa el patrón o modifica la expresión regular. Ejemplos: {examples}"
    )
df["hora"] = hour_text.astype(int)

# Objetivo continuo: logaritmo del área relativa del objeto.
df["area_rel"] = (
    (df[W_COL] * df[H_COL]) / (df["img_w"] * df["img_h"])
)
df["log_area_rel"] = np.log1p(df["area_rel"] * 1000)

# Entradas: solo mediciones de cabeza y hora.
# No se incluyen x, y, w, h del objeto porque se usan para calcular el objetivo.
df["head_area_rel"] = (
    (df["head_w"] * df["head_h"]) / (df["img_w"] * df["img_h"])
)
df["log_head_area_rel"] = np.log1p(df["head_area_rel"] * 1000)
df["head_cx_rel"] = df["head_cx"] / df["img_w"]
df["head_cy_rel"] = df["head_cy"] / df["img_h"]

FEATURES = [
    "log_head_area_rel",
    "head_w",
    "head_h",
    "head_cx_rel",
    "head_cy_rel",
    "hora",
]
TARGET = "log_area_rel"
GROUP_COL = IMAGE_COL

# Conservar solo filas completas para las variables que se modelarán.
df_model = df.dropna(subset=FEATURES + [TARGET, GROUP_COL]).copy()

X = df_model[FEATURES]
y = df_model[TARGET]
groups = df_model[GROUP_COL]

if groups.nunique() < 3:
    raise ValueError(
        "Hay muy pocas imágenes distintas para separar train/test y validar."
    )

# Separación agrupada: todas las observaciones de una imagen quedan juntas.
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()
groups_train = groups.iloc[train_idx].copy()
groups_test = groups.iloc[test_idx].copy()

# Comprobación explícita para detectar fuga entre imágenes.
assert set(groups_train).isdisjoint(set(groups_test))

# Validación cruzada agrupada, solo sobre entrenamiento.
n_splits = min(5, groups_train.nunique())
if n_splits < 2:
    raise ValueError("No hay suficientes imágenes en train para validación cruzada.")

cv = GroupKFold(n_splits=n_splits)

print(f"Filas totales: {len(df_model)}")
print(f"Imágenes totales: {groups.nunique()}")
print(f"Train: {len(X_train)} filas | {groups_train.nunique()} imágenes")
print(f"Test:  {len(X_test)} filas | {groups_test.nunique()} imágenes")
print(f"Validación cruzada: {n_splits} particiones agrupadas por imagen")
print(f"Características: {FEATURES}")
print(f"Objetivo: {TARGET}")

display(df_model[FEATURES + [TARGET]].head())

Filas totales: 66
Imágenes totales: 38
Train: 48 filas | 28 imágenes
Test:  18 filas | 10 imágenes
Validación cruzada: 5 particiones agrupadas por imagen
Características: ['log_head_area_rel', 'head_w', 'head_h', 'head_cx_rel', 'head_cy_rel', 'hora']
Objetivo: log_area_rel


,log_head_area_rel,head_w,head_h,head_cx_rel,head_cy_rel,hora,log_area_rel
0,4.983517,170,262,0.295312,0.504167,15,2.161286
1,4.854103,181,216,0.527344,0.531250,15,2.212800
2,4.528411,177,159,0.466406,0.165625,15,0.590027
3,5.491823,330,225,0.464062,0.765625,15,0.630690
4,5.773174,325,303,0.392969,0.684375,15,0.410660


## Modelo 1

## Modelo 2

## Modelo 3

## Modelo 4

## Modelo 5

## Modelo 6

# Comparativa

# Ajuste fino

# Modelo individual